## PPO Understanding

### Before PPO?
- While using classic Policy Gradient Algorithm training was unstable as it's objective was *"If an action produced high-reward, increase its probability"* as model follwed high-reward it can abruptly change its policy which lead to collapse in training, massive variance.

#### TRPO(Trust Region Policy Optimization)
- It introduced *"Don't allow the policy to move too far in one update"* meaning a threshold was set that policy can change upto this only, but implementing this was very complex.

### PPO(Proximal Policy Optimization)
- It introduced *"Let Gradient Descent improve the policy, but automatically ignore updates that try to change the policy too much."*
- PPO is an **on-policy algorithm** because it uses data collected by the current(or very recent) policy, and it prevents that policy from drifting too far while reusing the same batch.

## PPO Implementation

In [1]:
!pip install --upgrade vizdoom gymnasium wandb imageio opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.9/38.9 MB 32.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 58.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.0/318.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 23.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.8 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      Successfully uninstalled opencv-python-4.13.0.92
  Attempting uninstall: imageio
    Found existing installation: ImageIO 2.37.3
    Uninstalling ImageIO-2.37.3:
      Successfully uninstalled ImageIO-2.37.3
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.0
    Uninstalling gymnasium-1.2.0:
      Succe

In [2]:
import torch
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F
import numpy as np 
import gymnasium as gym 
from gymnasium import spaces 
import wandb
import cv2 
import os 
from vizdoom import gymnasium_wrapper 
import imageio

In [34]:
class Config:
    def __init__(self):
        self.env_name = "VizdoomDefendCenter-v1"
        self.total_timesteps = 2000000
        self.learning_rate = 2.5e-4
        self.gamma = 0.99
        self.gae_lambda = 0.95
        self.clip_epsilon = 0.2
        self.epochs = 4
        self.batch_size = 256
        self.buffer_size = 4096
        self.hidden_size = 512
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.frame_stack = 4
        self.frame_skip = 4
        self.entropy_coef = 0.03
        self.video_log_interval = 100
        self.num_actions = 3
        
config = Config()
# wandb.login(relogin=True)
wandb.init(project = "ppo-vizdoom", config = vars(config), mode = 'online')
        

In [27]:
class FrameSkipWrapper(gym.Wrapper):
    """
        Repeats action for 'skip' frames, accumulates rewards to speed up temporal credit assignment
        
    """
    def __init__(self, env, skip = 4):
        super().__init__(env)
        self.skip = skip
        self._obs_buffer = np.empty((2, *self.env.observation_space.shape), dtype = np.uint8)

    def step(self, action):
        total_reward = 0.0
        terminated, truncated = False, False
        accumulated_info = {}
        
        for i in range(self.skip):
            obs, reward, term, trunc, info = self.env.step(action)

            #Store observations for max-pooling on the last 2 steps of the step whose frames-we are skipping
            if i == self.skip - 2:
                self._obs_buffer[0] = obs
            if i == self.skip - 1:
                self._obs_buffer[1] = obs
            
            total_reward += reward
            terminated = terminated or term
            truncated = truncated or trunc

            #Preserve info dict updates across intermediate steps
            if isinstance(info, dict):
                accumulated_info.update(info)

            if terminated or truncated:
                break

        #Max-Pool over the last 2 frames to handle flickering sprites
        if i >= 1:
            max_obs = self._obs_buffer.max(axis = 0)
        else:
            max_obs = obs
            
        return max_obs, total_reward, terminated, truncated, info

In [14]:
class WandbVideoRecorder(gym.Wrapper):
    """ 
        This wrapper will grab the RBG frames at every step and at the end of an episode
        will stitch those frames into an .mp4
    """
    def __init__(self,env, interval = 100):
        super().__init__(env)   
        self.interval = interval 
        self.episode_count = 0
        self.recording = False 
        self.frames = []

    def _get_render_frame(self):
        """ 
            Helper to safely extract the RGB array from render()
        """
        frame = self.env.render()

        if isinstance(frame, dict):
            frame = frame.get('rgb', frame.get('screen', None))

        #Ensure it's a proper numpy array
        if frame is not None:
            frame = np.array(frame, dtype = np.uint8)
            
            # ViZDoom channels transpose check: (C, H, W) -> (H, W, C)
            if frame.ndim == 3 and frame.shape[0] in (1, 3, 4):
                frame = np.transpose(frame, (1, 2, 0))
        return frame
    
    def reset(self, **kwargs):
        if self.episode_count % self.interval == 0:
            self.recording = True 
            self.frames = []
        else:
            self.recording = False 
            
        obs, info = self.env.reset(**kwargs)
        
        #If recording, grab the first frame
        if self.recording:
            frame = self._get_render_frame()
            if frame is not None:
                self.frames.append(frame)
                
        return obs, info 
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        if self.recording:
            frame = self._get_render_frame()
            if frame is not None:
                self.frames.append(frame)

        done = terminated or truncated
        if done:
            if self.recording:
                self.save_and_log_video()
            else:
                self.episode_count += 1
                
        return obs, reward, terminated, truncated, info 
    
    def save_and_log_video(self):
        if self.recording and len(self.frames) > 0:
            video_path = f"vizdoom_ep_{self.episode_count}.mp4"
            imageio.mimsave(video_path, self.frames, fps = 30)
        
            wandb.log({
                "gameplay_video": wandb.Video(video_path, fps = 30, format = "mp4"),
                "episode": self.episode_count
            })
            print(f"-------Successfully logged videos for Episode {self.episode_count}----")
        self.episode_count += 1
        self.recording = False 
        self.frames = []
        

In [15]:
class ImagePreprocessingWrapper(gym.Wrapper):
    """ 
        A wrapper to perform image pre-processing operations 
    """
    def __init__(self, env, frame_stack = 4):
        super().__init__(env)
        self.frame_stack = frame_stack
        #Observation Shape
        self.obs_shape = (84, 84)
        
        #Override the Observation_Space inorder to match stacked grayscale frames
        self.observation_space = spaces.Box(low = 0, high = 1.0, shape = (frame_stack, 84, 84), dtype = np.float32)
        self.frames = []
        
    def _preprocess(self, obs):
        if isinstance(obs, dict):
            obs = obs.get("screen", obs.get("rgb", None)) # Extract the image array, ignore the rest
            if obs is None:
                raise KeyError("Observation dict missing both 'screen' and 'rgb' keys")
            
        # 2. If for some reason it's still a tuple/list, grab the first element
        if isinstance(obs, (tuple, list)):
            obs = obs[0]

        
        #Observation comes in as (240, 320, 3) numpy array i.e. (width, height, RGB)
        obs = np.array(obs, dtype = np.uint8)

        # 3. Transpose (C, H, W) -> (H, W, C) if needed
        if obs.ndim == 3 and obs.shape[0] in (1, 3, 4):
            obs = np.transpose(obs, (1, 2, 0))
            
        # 4. Convert to Grayscale
        if obs.ndim == 3 and obs.shape[2] == 3:
            gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        elif obs.ndim == 3 and obs.shape[2] == 1:
            gray = obs.squeeze(-1)
        else:
            gray = obs
        
        #RESIZE TO 84 x 84
        resized = cv2.resize(gray, self.obs_shape, interpolation = cv2.INTER_AREA)
        
        #NORMALIZE to [0, 1]
        normalized = resized.astype(np.float32) / 255.0
        
        return normalized
    
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        processed = self._preprocess(obs)
        
        #Fill frame stack with the first frame
        self.frames = [processed for _ in range(self.frame_stack)]
        return np.array(self.frames, dtype = np.float32), info 
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        processed = self._preprocess(obs)
        
        #Append new frames, remove the oldest
        self.frames.append(processed)
        self.frames.pop(0)
        
        return np.array(self.frames, dtype = np.float32), reward, terminated, truncated, info
    

In [16]:
class RewardShaper(gym.Wrapper):
    def __init__(self, env, fire_action_id = 2, ammo_key = "ammo", health_key = "health", initial_ammo = 26, initial_health = 100):
        super().__init__(env)
        self.fire_action_id = fire_action_id
        self.ammo_key = ammo_key
        self.health_key = health_key

        self.initial_ammo = initial_ammo
        self.initial_health = initial_health

        self.previous_ammo = initial_ammo
        self.previous_health = initial_health

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        #Safely extract starting state variables if present in the env
        if isinstance(info, dict):
            self.previous_ammo = info.get(self.ammo_key, self.initial_ammo)
            self.previous_health = info.get(self.health_key, self.initial_health)
        else:
            self.previous_ammo = self.initial_ammo
            self.previous_health = self.initital_health
    
        return obs, info 

    def step(self, action):
        obs, original_reward, terminated, truncated, info = self.env.step(action)

        shaped_reward = 0.0

        #1 Kill Reward(VizDoom +1.0, we will give +2.0)
        if original_reward > 0:
            shaped_reward += 2.0

        #Extract current state variables safely from info dict
        current_ammo = self.previous_ammo
        current_health = self.previous_health

        if isinstance(info, dict):
            current_ammo = info.get(self.ammo_key, self.previous_ammo)
            current_health = info.get(self.health_key, self.previous_health)

        #2 Ammo-Rewards (We Penalize missed/wasted shots)
        ammo_used = self.previous_ammo - current_ammo
        if ammo_used > 0:
            if original_reward <= 0:
                #Shot fired but no monster died --> We penalize missed shot
                shaped_reward -= 0.02 * ammo_used
            else:
                #Shot resulted in a kill --> We give Tiny efficiency bonus
                shaped_reward += 0.01

        #3 Health Loss Penalty 
        #If the agent stays stationary, demons attack from behind
        #So peanlizing health_loss forces the policy to rotate and check its surroundings.
        health_loss = self.previous_health - current_health
        if health_loss > 0:
            shaped_reward -= 0.05 * health_lost

        self.previous_ammo   = current_ammo
        self.previous_health = current_health

        return obs, shaped_reward, terminated, truncated, info

In [17]:
class ActorCritic(nn.Module):
    def __init__(self, num_actions):
        super(ActorCritic, self).__init__()
        
        #------------CNN Based Feature Extractor-------------
        #Input: (4, 84, 84) -> Output: (64, 7, 7) --> Flatten --> 3136
        self.shared = nn.Sequential(
            nn.Conv2d(in_channels = 4, out_channels = 32, kernel_size = 8, stride = 4),
            nn.ReLU(),
            nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 4, stride = 2),
            nn.ReLU(),
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = 3, stride = 1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, config.hidden_size),
            nn.ReLU()
        )
        
        self.actor = nn.Linear(config.hidden_size, num_actions)
        # self.actor_logstd = nn.Parameter(torch.zeros(action_dim))
        
        self.critic = nn.Linear(config.hidden_size, 1)
        
    
    def forward(self, x):
        features = self.shared(x) #Shape: [Batch, 4, 84, 84]
        logits = self.actor(features)
        value = self.critic(features)
        return logits, value
    
    def get_action(self, obs):
        #Obs comes in as an Numpy Array
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(config.device)  #We also add batch_dim SHAPE:[1, 4, 84, 84]
        logits, value = self.forward(obs_tensor)
        #Use categorical distribution instead of Normal
        dist = torch.distributions.Categorical(logits = logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        
        action = action.squeeze(0) #Remove batch dim from env
        return action.cpu().detach().numpy(), log_prob.cpu().detach().item(), value.cpu().detach().item()
    
    def evaluate(self, obs, action):
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits = logits)
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return log_prob, value.squeeze(-1), entropy    
        

In [18]:
class RolloutBuffer:
    def __init__(self):
        self.obs = []
        self.actions = []
        self.rewards = []          #Rewards shaped for PPO GAE computation
        self.raw_rewards = []      #Rewards returned from the environment
        self.dones = []
        self.log_probs = []
        self.values = []
        
    def add(self, obs, action, reward,raw_reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.raw_rewards.append(raw_reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)
        
    def get(self):
        data = {
            "obs": torch.FloatTensor(np.array(self.obs)).to(config.device),
            "actions": torch.LongTensor(np.array(self.actions)).to(config.device),
            "rewards": torch.FloatTensor(np.array(self.rewards)).to(config.device),
            "raw_rewards": np.array(self.raw_rewards),
            "dones": torch.FloatTensor(np.array(self.dones)).to(config.device),
            "log_probs": torch.FloatTensor(np.array(self.log_probs)).to(config.device),
            "values": torch.FloatTensor(np.array(self.values)).to(config.device)
        }
        
        self.clear()
        return data 
    
    def clear(self):
        self.obs, self.actions, self.rewards, self.raw_rewards = [], [], [], []
        self.dones, self.log_probs, self.values = [], [], []

#### GAE(Generalized Advantage Estimation)
- Temporal Difference Error represents the immediate surprise in reward plus discounted future value.
- GAE balances variance and bias by taking an exponentially weighted average of k-step advantages.

In [19]:
def compute_gae(buffer_data, last_value):
    #Get the trajectory data collected during the rollout
    rewards = buffer_data["rewards"]    #This is the immediate rewards r_t
    values = buffer_data["values"]      #This is the Critic's esitmated state values V(s_t)
    dones = buffer_data["dones"]        #This is the termination flags
    
    #Initialize the advantage tensor with zeros matching the shape of the Rewards tensor
    advantages = torch.zeros_like(rewards).to(config.device)
    last_gae = 0
    
    #Iterate backwards through time: t = T-1, T-2, T-3,.....,0
    for t in reversed(range(len(rewards))):
 
        #Determine the Value{s_{t+1}} for the next step
        if t == len(rewards) - 1:
            next_value = last_value     #Value of the state reached after the final step
        else:
            next_value = values[t + 1]
        
        #Calculate Temporal Difference(TD) error: delta_at_t = reward_at_t + (gamma * Value_at_step_t+1 *(1 - done_at_t)) - Value_at_step_t
        delta = rewards[t] + config.gamma * next_value * (1 - dones[t]) - values[t]
        last_gae = delta + config.gamma * config.gae_lambda * (1 - dones[t]) * last_gae
        advantages[t] = last_gae
        
    returns = advantages + values 
    return advantages , returns  

def ppo_update(policy, optimizer, buffer_data, advantages, returns):
    obs = buffer_data["obs"]
    actions = buffer_data["actions"]
    old_log_probs = buffer_data["log_probs"]
    
    #Normalize Advantages to have mean 0 and standard deviation 1 to keep gradient updates consistent
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    dataset_size = len(obs)
    indices = np.arange(dataset_size)
    
    policy_losses, value_losses, entropies = [], [], []
    
    #Mini-Batch Optimization Loop
    for _ in range(config.epochs):
        np.random.shuffle(indices)
        
        #Slice the dataset into mini-batches
        for start in range(0, dataset_size, config.batch_size):
            end = start + config.batch_size
            batch_idx = indices[start:end]
            
            #Slice mini-batch data 
            b_obs = obs[batch_idx]
            b_actions = actions[batch_idx]
            b_old_log_probs = old_log_probs[batch_idx]
            b_advantages = advantages[batch_idx]
            b_returns = returns[batch_idx]
            
            log_prob, value, entropy = policy.evaluate(b_obs, b_actions)
            
            #Compute probability-ratio r_t(theta)
            ratio = torch.exp(log_prob - b_old_log_probs)
            
            #Unclipped objective element
            surr1 = ratio * b_advantages 
            
            #Clipped objective element
            surr2 = torch.clamp(ratio, 1 - config.clip_epsilon, 1 + config.clip_epsilon) * b_advantages
            
            #PPO Clipped Surrogate Loss
            policy_loss = -torch.min(surr1, surr2).mean()
            
            #Value function i.e Critic Loss using MSE
            value_loss = nn.MSELoss()(value, b_returns)
            
            #Mean Policy Entropy
            entropy_loss = entropy.mean()
            
            #Combined Loss Function in which model will try to minimize the policy,value loss while maximizing the entropy
            loss = policy_loss + 0.5 * value_loss - config.entropy_coef * entropy_loss
            
            optimizer.zero_grad()
            loss.backward()
            
            #Gradient clipping
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()
            
            policy_losses.append(policy_loss.item())
            value_losses.append(value_loss.item())
            entropies.append(entropy_loss.item())
            
            
    return np.mean(policy_losses), np.mean(value_losses), np.mean(entropies)        

In [35]:
def train():
    env = gym.make(config.env_name, render_mode = "rgb_array")
    #Apply Image Preprocessing wrapper 
    env = ImagePreprocessingWrapper(env, frame_stack = config.frame_stack)
    #Apply Frame Skipp first (Action repeat at the engine level)
    env = FrameSkipWrapper(env, skip = config.frame_skip)
    #Apply Reward Shaper (Calculates penalties on aggregated step deltas)
    env = RewardShaper(env)
    #Apply Video Recording
    env = WandbVideoRecorder(env, interval = config.video_log_interval)
    
    policy = ActorCritic(config.num_actions).to(config.device)
    optimizer = optim.Adam(policy.parameters(), lr = config.learning_rate)
    buffer = RolloutBuffer()
    
    obs, info = env.reset()
    shaped_episode_reward = 0
    raw_episode_reward = 0
    episode_length = 0
    episode_count = 0

    #Metrics: Action Distribution Traccking
    # action_counts = {a: 0 for a in range(config.num_actions)}

    #Metrics: Episode metrics history for buffer logging
    recent_raw_returns = []
    recent_shaped_returns = []
    recent_kills = []
    
    print(f"Starting VizDoom training for {config.total_timesteps} timesteps...")
    print(f"Using Device: {config.device}")
    
    for timestep in range(1, config.total_timesteps + 1):

        #Linear Learning-Rate Decay
        frac = 1.0 - (timestep - 1.0) / config.total_timesteps
        lrnow = frac * config.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow

        #Get the Action from Policy Network
        action, log_prob, value = policy.get_action(obs)

        #Metrics: Update the metrics count
        # action_counts[action] += 1

        #Take a Step in Environment        
        next_obs, shaped_reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        #Extract Raw unshaped environment reward(if stored in info by wrapper)
        raw_reward = info.get("raw_reward", shaped_reward)

        #Store in buffer
        buffer.add(obs, action, shaped_reward,raw_reward, done, log_prob, value)
        
        obs = next_obs 
        shaped_episode_reward += shaped_reward 
        raw_episode_reward += raw_reward
        episode_length += 1 
        
        #Update PPO
        if timestep % config.buffer_size == 0:
            with torch.no_grad():
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(config.device)
                _, last_value = policy.forward(obs_tensor)
                last_value = last_value.cpu().item()
                
            buffer_data = buffer.get()
            advantages, returns = compute_gae(buffer_data, last_value)
            
            avg_pol_loss, avg_val_loss, avg_entropy = ppo_update(policy, optimizer, buffer_data, advantages, returns)
            
            wandb.log({
                "timesteps": timestep,
                "train/policy_loss": avg_pol_loss,
                "train/value_loss": avg_val_loss,
                "train/entropy": avg_entropy,
            })

        if timestep % 100000 == 0:
            ckpt_path = f"ppo_vizdoom_step_{timestep}.pth"
            torch.save(
                {
                    "timestep": timestep,
                    "model_state_dict": policy.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                },
                ckpt_path
            )
            print(f"Saved checkpoint at step: {timestep} -> {ckpt_path}")
        if done:
            episode_count += 1
            wandb.log({
                "episode": episode_count,
                "episode_reward" : shaped_episode_reward,
                "raw_reward": raw_episode_reward,
                "episode_length": episode_length
            })
            
            if episode_count % 10 == 0:
                print(f"Timestep: {timestep} | Episode: {episode_count} | Reward: {shaped_episode_reward:.2f}")
                
            obs, _ = env.reset()
            episode_reward = 0
            episode_length = 0
            
    final_path = "ppo_vizdoom_final.pth"
    torch.save(policy.state_dict(), final_path)
    print("Final Model Saved!")
    env.close()
    wandb.finish()
    print(f"VizDoom Training Completed")

In [ ]:
train()

Starting VizDoom training for 2000000 timesteps...
Using Device: cuda


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


-------Successfully logged videos for Episode 0----
Timestep: 824 | Episode: 10 | Reward: 28.00
Timestep: 1572 | Episode: 20 | Reward: 58.00
Timestep: 2377 | Episode: 30 | Reward: 88.00
Timestep: 3132 | Episode: 40 | Reward: 122.00
Timestep: 3944 | Episode: 50 | Reward: 152.00
Timestep: 4709 | Episode: 60 | Reward: 182.00
Timestep: 5446 | Episode: 70 | Reward: 208.00
Timestep: 6221 | Episode: 80 | Reward: 234.00
Timestep: 6959 | Episode: 90 | Reward: 264.00
Timestep: 7663 | Episode: 100 | Reward: 288.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


-------Successfully logged videos for Episode 100----
Timestep: 8435 | Episode: 110 | Reward: 318.00
Timestep: 9179 | Episode: 120 | Reward: 344.00
Timestep: 9948 | Episode: 130 | Reward: 370.00
Timestep: 10663 | Episode: 140 | Reward: 396.00
Timestep: 11422 | Episode: 150 | Reward: 418.00
Timestep: 12174 | Episode: 160 | Reward: 446.00
Timestep: 12933 | Episode: 170 | Reward: 478.00
Timestep: 13738 | Episode: 180 | Reward: 514.00
Timestep: 14503 | Episode: 190 | Reward: 542.00
Timestep: 15273 | Episode: 200 | Reward: 566.00


...

-------Successfully logged videos for Episode 23100----
Timestep: 1747663 | Episode: 23110 | Reward: 66752.00
Timestep: 1748361 | Episode: 23120 | Reward: 66778.00
Timestep: 1749086 | Episode: 23130 | Reward: 66802.00
Timestep: 1749827 | Episode: 23140 | Reward: 66830.00
Timestep: 1750583 | Episode: 23150 | Reward: 66854.00
Timestep: 1751347 | Episode: 23160 | Reward: 66882.00
Timestep: 1751995 | Episode: 23170 | Reward: 66908.00
Timestep: 1752646 | Episode: 23180 | Reward: 66928.00
Timestep: 1753459 | Episode: 23190 | Reward: 66958.00
Timestep: 1754283 | Episode: 23200 | Reward: 66990.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


-------Successfully logged videos for Episode 23200----
Timestep: 1755009 | Episode: 23210 | Reward: 67014.00
Timestep: 1755784 | Episode: 23220 | Reward: 67038.00
Timestep: 1756565 | Episode: 23230 | Reward: 67068.00
Timestep: 1757396 | Episode: 23240 | Reward: 67104.00
Timestep: 1758150 | Episode: 23250 | Reward: 67128.00
Timestep: 1758943 | Episode: 23260 | Reward: 67156.00
Timestep: 1759749 | Episode: 23270 | Reward: 67190.00
Timestep: 1760447 | Episode: 23280 | Reward: 67216.00
Timestep: 1761208 | Episode: 23290 | Reward: 67246.00
Timestep: 1761944 | Episode: 23300 | Reward: 67276.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


-------Successfully logged videos for Episode 23300----
Timestep: 1762754 | Episode: 23310 | Reward: 67304.00
Timestep: 1763545 | Episode: 23320 | Reward: 67334.00
Timestep: 1764258 | Episode: 23330 | Reward: 67360.00
Timestep: 1765028 | Episode: 23340 | Reward: 67394.00
Timestep: 1765779 | Episode: 23350 | Reward: 67422.00
Timestep: 1766569 | Episode: 23360 | Reward: 67454.00
Timestep: 1767453 | Episode: 23370 | Reward: 67490.00


In [ ]:
wandb.finish()